# Notebook 3: Classification Training

Train all 24 classification experiments:
- 4 feature types: de_LDS, de_movingAve, psd_LDS, psd_movingAve
- 6 models: MLP, CNN Single, CNN Multi, EEGNet, DE-CNN, Transformer
- Task: Binary classification (alert vs fatigued)

**Output**: All model checkpoints + classification_results.json

In [1]:
# Import from Notebook 2
import sys
sys.path.append('/home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/Codes')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import json
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

PyTorch version: 2.9.1+cu128
Using device: cuda
GPU: NVIDIA GeForce RTX 5080
CUDA Version: 12.8


## 1. Load Model Definitions from Notebook 2

In [2]:
# Since we can't directly import from notebooks, we'll redefine the essential classes
# Or you can run Notebook 2 first and use %run magic command

# For now, let's use %run to execute Notebook 2
%run 2_baseline_models.ipynb

PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080
CUDA Version: 12.8
Dataset classes defined successfully!
MLPBaseline defined!
CNN2DSingleFrame defined!
CNN2DMultiFrame defined!
EEGNet defined!
DECNN defined!
ChannelTransformer defined!

Testing All Models

MLP Baseline:
  Input shape:  torch.Size([4, 85])
  Output shape: torch.Size([4, 2])
  Total params: 21,410
  Trainable:    21,410
  ✓ Test passed!

CNN2D Single Frame:
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 23,298
  Trainable:    23,298
  ✓ Test passed!

CNN2D Multi-Frame (T=5):
  Input shape:  torch.Size([4, 5, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 101,378
  Trainable:    101,378
  ✓ Test passed!

EEGNet:
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 730
  Trainable:    730
  ✓ Test passed!

DE-CNN (VIGNet-style):
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])


## 2. Configuration

In [3]:
# Paths
BASE_DIR = Path('/home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/SEED-VIG')
SAVE_DIR = Path('/home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/Models/baselines/classification')
RESULTS_DIR = Path('/home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/results/baselines')

# Experiment settings
FEATURE_KEYS = ['de_LDS', 'de_movingAve', 'psd_LDS', 'psd_movingAve']
MODEL_NAMES = ['mlp', 'cnn_single', 'cnn_multi', 'eegnet', 'decnn', 'transformer']

PERCLOS_THRESHOLD = 0.35
TRAIN_RATIO = 0.80
SEED = 42

# Model-specific configs
CONFIG = {
    'mlp': {'batch_size': 256, 'lr': 1e-3, 'weight_decay': 1e-4, 'epochs': 50},
    'cnn_single': {'batch_size': 128, 'lr': 1e-3, 'weight_decay': 1e-4, 'epochs': 50},
    'cnn_multi': {'batch_size': 64, 'lr': 1e-3, 'weight_decay': 1e-4, 'epochs': 50, 'T_seq': 5, 'step': 2},
    'eegnet': {'batch_size': 128, 'lr': 1e-3, 'weight_decay': 1e-4, 'epochs': 100},
    'decnn': {'batch_size': 64, 'lr': 5e-4, 'weight_decay': 1e-4, 'epochs': 75},
    'transformer': {'batch_size': 64, 'lr': 5e-4, 'weight_decay': 1e-4, 'epochs': 75}
}

print("Configuration loaded!")

Configuration loaded!


## 3. Training Functions

In [4]:
def get_weighted_loss(y_train):
    """
    Compute weighted CrossEntropyLoss for class imbalance.
    
    Args:
        y_train: (N,) training labels
    
    Returns:
        Weighted loss function
    """
    unique, counts = np.unique(y_train, return_counts=True)
    weights = 1.0 / counts
    weights = weights / weights.sum() * len(unique)  # Normalize
    weights = torch.from_numpy(weights).float()
    
    print(f"  Class weights: {dict(zip(unique.tolist(), weights.tolist()))}")
    return nn.CrossEntropyLoss(weight=weights)


def evaluate_classification(model, data_loader, device):
    """
    Evaluate classification model on a dataset.
    
    Returns:
        dict with keys: acc, f1, precision, recall
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(device)
            outputs = model(xb)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(yb.numpy())
    
    # Concatenate all batches
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    
    # Compute metrics
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    precision = precision_score(y_true, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_true, y_pred, average='binary', zero_division=0)
    
    return {
        'acc': float(acc),
        'f1': float(f1),
        'precision': float(precision),
        'recall': float(recall)
    }


def train_classification(model, train_loader, val_loader, test_loader, y_train, num_epochs, lr, weight_decay, checkpoint_path, device):
    """Train classification model with proper train/val/test split."""
    model = model.to(device)
    criterion = get_weighted_loss(y_train).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    best_f1 = 0.0
    best_metrics = None
    
    for epoch in range(1, num_epochs + 1):
        # Training
        model.train()
        train_loss = 0.0
        
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            
            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation (for model selection)
        val_metrics = evaluate_classification(model, val_loader, device)
        
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{num_epochs} | Loss: {train_loss/len(train_loader):.4f} | "
                  f"Val Acc: {val_metrics['acc']*100:.2f}% | Val F1: {val_metrics['f1']:.3f}")
        
        # Save best model based on validation F1
        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            torch.save(model.state_dict(), checkpoint_path)
    
    # Load best model and evaluate on test set (final evaluation only)
    model.load_state_dict(torch.load(checkpoint_path))
    test_metrics = evaluate_classification(model, test_loader, device)
    
    # Clear CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return test_metrics

print("Training functions defined!")

Training functions defined!


## 4. Model Factory

In [5]:
def create_model(model_name, num_outputs=2):
    """Create model instance."""
    if model_name == 'mlp':
        return MLPBaseline(input_size=85, num_outputs=num_outputs)
    elif model_name == 'cnn_single':
        return CNN2DSingleFrame(num_outputs=num_outputs)
    elif model_name == 'cnn_multi':
        return CNN2DMultiFrame(T_seq=5, num_outputs=num_outputs)
    elif model_name == 'eegnet':
        return EEGNet(num_outputs=num_outputs)
    elif model_name == 'decnn':
        return DECNN(num_outputs=num_outputs)
    elif model_name == 'transformer':
        return ChannelTransformer(num_outputs=num_outputs)
    else:
        raise ValueError(f"Unknown model: {model_name}")

print("Model factory defined!")

Model factory defined!


## 5. Single Experiment Runner

In [6]:
def run_single_classification_experiment(feature_key, model_name):
    """Run one complete classification experiment with 80-10-10 split."""
    print(f"\n{'='*70}")
    print(f"Experiment: Classification | {feature_key} | {model_name}")
    print(f"{'='*70}")
    
    # 1. Load data
    print("Loading data...")
    X_all, y_all, subj_all = load_seedvig_5band_all_features(BASE_DIR, feature_key)
    
    # 2. Normalize
    print("Normalizing features...")
    X_all = normalize_features_subjectwise(X_all, subj_all)
    
    # 3. Create binary labels
    y_binary = (y_all >= PERCLOS_THRESHOLD).astype(np.int32)
    print(f"Binary labels: 0={np.sum(y_binary==0)}, 1={np.sum(y_binary==1)}")
    
    # 4. Split subjects (80-10-10)
    train_mask, val_mask, test_mask = subject_wise_split(subj_all, train_ratio=0.80, val_ratio=0.10, seed=SEED)
    
    X_train, X_val, X_test = X_all[train_mask], X_all[val_mask], X_all[test_mask]
    y_train, y_val, y_test = y_binary[train_mask], y_binary[val_mask], y_binary[test_mask]
    subj_train, subj_val, subj_test = subj_all[train_mask], subj_all[val_mask], subj_all[test_mask]
    
    # 5. Create datasets
    print(f"Creating datasets for {model_name}...")
    cfg = CONFIG[model_name]
    
    if model_name == 'mlp':
        train_ds = SEEDVIGSingleFrameDataset(X_train, y_train, flatten=True, regression=False)
        val_ds = SEEDVIGSingleFrameDataset(X_val, y_val, flatten=True, regression=False)
        test_ds = SEEDVIGSingleFrameDataset(X_test, y_test, flatten=True, regression=False)
    elif model_name in ['cnn_single', 'eegnet', 'decnn', 'transformer']:
        train_ds = SEEDVIGSingleFrameDataset(X_train, y_train, flatten=False, regression=False)
        val_ds = SEEDVIGSingleFrameDataset(X_val, y_val, flatten=False, regression=False)
        test_ds = SEEDVIGSingleFrameDataset(X_test, y_test, flatten=False, regression=False)
    elif model_name == 'cnn_multi':
        train_ds = SEEDVIGMultiFrameDataset(X_train, y_train, subj_train, 
                                           T_seq=cfg['T_seq'], step=cfg['step'], regression=False)
        val_ds = SEEDVIGMultiFrameDataset(X_val, y_val, subj_val, 
                                         T_seq=cfg['T_seq'], step=cfg['step'], regression=False)
        test_ds = SEEDVIGMultiFrameDataset(X_test, y_test, subj_test, 
                                          T_seq=cfg['T_seq'], step=cfg['step'], regression=False)
    
    # 6. Create loaders (num_workers=0 for notebook safety)
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=0)
    
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)} samples")
    
    # 7. Create model
    model = create_model(model_name, num_outputs=2)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")
    
    # 8. Train
    checkpoint_path = SAVE_DIR / feature_key / f"{model_name}_best.pth"
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"Training for {cfg['epochs']} epochs...")
    metrics = train_classification(
        model, train_loader, val_loader, test_loader, y_train,
        num_epochs=cfg['epochs'],
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
        checkpoint_path=checkpoint_path,
        device=device
    )
    
    print(f"\nFinal Test Metrics:")
    print(f"  Accuracy:  {metrics['acc']*100:.2f}%")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    
    return metrics

print("Experiment runner defined!")

Experiment runner defined!


## 6. Run All 24 Classification Experiments

In [7]:
# Initialize results dictionary
all_results = {}

# Counter
experiment_count = 0
total_experiments = len(FEATURE_KEYS) * len(MODEL_NAMES)

print(f"\n" + "="*70)
print(f"Starting {total_experiments} Classification Experiments")
print(f"="*70)

for feature_key in FEATURE_KEYS:
    all_results[feature_key] = {}
    
    for model_name in MODEL_NAMES:
        experiment_count += 1
        print(f"\n\n>>> Progress: {experiment_count}/{total_experiments} experiments completed")
        
        try:
            metrics = run_single_classification_experiment(feature_key, model_name)
            all_results[feature_key][model_name] = metrics
            
        except Exception as e:
            print(f"\n!!! ERROR in {feature_key} / {model_name}: {e}")
            all_results[feature_key][model_name] = {'error': str(e)}

print(f"\n\n" + "="*70)
print(f"All {total_experiments} Classification Experiments Complete!")
print(f"="*70)


Starting 24 Classification Experiments


>>> Progress: 1/24 experiments completed

Experiment: Classification | de_LDS | mlp
Loading data...
Loaded 23 sessions (each treated as separate subject)
  X shape: (20355, 17, 5)
  y shape: (20355,)
  Unique sessions: 23
Normalizing features...
Normalization complete (subject-wise z-score)
Binary labels: 0=7405, 1=12950
Subject split (80-10-10):
  Train: 18 subjects ([19, 11, 18, 23, 17, 16, 8, 7, 10, 4, 1, 20, 13, 6, 12, 15, 22, 3]), 15930 samples
  Val:   2 subjects ([5, 21]), 1770 samples
  Test:  3 subjects ([2, 14, 9]), 2655 samples
Creating datasets for mlp...
Train: 15930, Val: 1770, Test: 2655 samples
Model parameters: 21,410
Training for 50 epochs...
  Class weights: {0: 1.3010671138763428, 1: 0.6989328265190125}
  Epoch   1/50 | Loss: 0.5597 | Val Acc: 68.53% | Val F1: 0.771
  Epoch  10/50 | Loss: 0.2287 | Val Acc: 65.65% | Val F1: 0.759
  Epoch  20/50 | Loss: 0.1515 | Val Acc: 71.86% | Val F1: 0.813
  Epoch  30/50 | Loss: 0.1091 | V

## 7. Save Results

In [8]:
# Save results to JSON
results_file = RESULTS_DIR / 'classification_results.json'
results_file.parent.mkdir(parents=True, exist_ok=True)

# Convert numpy types to Python types for JSON serialization
def convert_to_serializable(obj):
    if isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    return obj

all_results_serializable = convert_to_serializable(all_results)

with open(results_file, 'w') as f:
    json.dump(all_results_serializable, f, indent=2)

print(f"\nResults saved to: {results_file}")


Results saved to: /home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/results/baselines/classification_results.json


## 8. Summary Table

In [9]:
import pandas as pd

# Create summary table
summary_data = []

for feature_key in FEATURE_KEYS:
    for model_name in MODEL_NAMES:
        if 'error' not in all_results[feature_key][model_name]:
            metrics = all_results[feature_key][model_name]
            summary_data.append({
                'Feature': feature_key,
                'Model': model_name,
                'Accuracy': f"{metrics['acc']*100:.2f}%",
                'F1': f"{metrics['f1']:.4f}",
                'Precision': f"{metrics['precision']:.4f}",
                'Recall': f"{metrics['recall']:.4f}"
            })
        else:
            summary_data.append({
                'Feature': feature_key,
                'Model': model_name,
                'Accuracy': 'ERROR',
                'F1': 'ERROR',
                'Precision': 'ERROR',
                'Recall': 'ERROR'
            })

df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("Classification Results Summary")
print("="*70)
print(df.to_string(index=False))

# Save to CSV
csv_file = RESULTS_DIR / 'classification_summary.csv'
df.to_csv(csv_file, index=False)
print(f"\nSummary saved to: {csv_file}")


Classification Results Summary
      Feature       Model Accuracy     F1 Precision Recall
       de_LDS         mlp   67.50% 0.6821    0.5879 0.8123
       de_LDS  cnn_single   67.98% 0.7087    0.5816 0.9070
       de_LDS   cnn_multi   49.13% 0.6213    0.4570 0.9701
       de_LDS      eegnet   68.74% 0.6706    0.6123 0.7412
       de_LDS       decnn   53.67% 0.6295    0.4794 0.9167
       de_LDS transformer   69.15% 0.6861    0.6093 0.7851
 de_movingAve         mlp   65.61% 0.6773    0.5672 0.8404
 de_movingAve  cnn_single   44.86% 0.6077    0.4375 0.9947
 de_movingAve   cnn_multi   49.43% 0.6290    0.4595 0.9965
 de_movingAve      eegnet   67.27% 0.6669    0.5922 0.7632
 de_movingAve       decnn   44.52% 0.6054    0.4358 0.9912
 de_movingAve transformer   63.20% 0.6446    0.5507 0.7772
      psd_LDS         mlp   60.15% 0.6316    0.5237 0.7956
      psd_LDS  cnn_single   44.52% 0.5950    0.4333 0.9491
      psd_LDS   cnn_multi   43.61% 0.5906    0.4294 0.9455
      psd_LDS      eegne

## 9. Best Models per Feature Type

In [10]:
print("\n" + "="*70)
print("Best Model per Feature Type (by F1 score)")
print("="*70)

for feature_key in FEATURE_KEYS:
    best_f1 = 0
    best_model = None
    
    for model_name in MODEL_NAMES:
        if 'error' not in all_results[feature_key][model_name]:
            f1 = all_results[feature_key][model_name]['f1']
            if f1 > best_f1:
                best_f1 = f1
                best_model = model_name
    
    if best_model:
        metrics = all_results[feature_key][best_model]
        print(f"\n{feature_key}:")
        print(f"  Best Model: {best_model}")
        print(f"  Accuracy:   {metrics['acc']*100:.2f}%")
        print(f"  F1 Score:   {metrics['f1']:.4f}")

print("\n" + "="*70)
print("Classification Training Complete!")
print("All models saved in: Models/baselines/classification/")
print("="*70)


Best Model per Feature Type (by F1 score)

de_LDS:
  Best Model: cnn_single
  Accuracy:   67.98%
  F1 Score:   0.7087

de_movingAve:
  Best Model: mlp
  Accuracy:   65.61%
  F1 Score:   0.6773

psd_LDS:
  Best Model: mlp
  Accuracy:   60.15%
  F1 Score:   0.6316

psd_movingAve:
  Best Model: eegnet
  Accuracy:   63.50%
  F1 Score:   0.6490

Classification Training Complete!
All models saved in: Models/baselines/classification/
